# Complex-Time Anharmonic Propagator — physical-cycle phase + descent loss

This notebook computes the zero-endpoint complex-time anharmonic-oscillator
propagator with `N = 16` for three independent random seeds and
`lambda = 0, 0.7, 2.4`.

The contour is **structurally constrained to the physical homology class**:

- it passes through the zero saddle exactly;
- close to the saddle it starts from the exact harmonic thimble;
- for `lambda > 0`, its tails are fixed in the lower quartic Stokes wedge
  continuously connected to the original real contour;
- the neural correction vanishes at infinity, so training cannot jump to the
  incorrect upper Stokes cycle that produces the nearly lambda-independent answer;
- the full complex Jacobian is used in estimation (and during training for the
  log-weight-variance notebook).

A separate harmonic-basis calculation supplies the fixed-N reference. Every
neural result is written to CSV with its error and a PASS/FAIL flag. Loss curves,
model checkpoints and a progress log are saved immediately after each run.


In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import time
import traceback
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
import torch
import torch.nn as nn

ip = get_ipython() if "get_ipython" in globals() else None
if ip is not None:
    ip.run_line_magic("matplotlib", "inline")

torch.set_default_dtype(torch.float64)

# -----------------------------------------------------------------------------
# One-click overnight configuration
# -----------------------------------------------------------------------------
METHOD = "phase_descent"  # "phase_descent" or "logweight_variance"

N = 16
T = 1.0
ETA = 0.05
MASS = 1.0
OMEGA = 1.0
X_I = 0.0
X_F = 0.0

LAMBDAS = [0.0, 0.7, 2.4]
SEEDS = [52, 53, 54]

RANK = 32
TAIL_RADIUS_FACTOR = 2.0
TAIL_POWER = 8
assert TAIL_POWER >= 4 and TAIL_POWER % 2 == 0

if METHOD == "phase_descent":
    EPOCHS = 15_000
    BATCH = 768
    LR = 2.0e-3
else:
    EPOCHS = 15_000
    BATCH = 384
    LR = 1.0e-3

# lambda=0 starts on the exact finite-N harmonic thimble; only a short check is needed.
EPOCHS_LAMBDA0 = 100
N_MC = 400_000
MC_CHUNK = 2_000
LOG_EVERY = 500
GRAD_CLIP = 5.0
RELATIVE_ERROR_TOL = 0.05
REFERENCE_BASIS = 300

# A broad component is essential for detecting contributions missed by a central
# Gaussian. The exact mixture density is included in every importance weight.
TRAIN_SCALES = (1.0, 1.5, 2.25)
TRAIN_PROBS = (0.80, 0.15, 0.05)
MC_SCALES = (1.0, 1.5, 2.25)
MC_PROBS = (0.80, 0.15, 0.05)

# Set THIMBLE_QUICK_TEST=1 only for a short installation/syntax check.
QUICK_TEST = os.environ.get("THIMBLE_QUICK_TEST", "0") == "1"
if QUICK_TEST:
    EPOCHS = 3
    EPOCHS_LAMBDA0 = 2
    BATCH = 16
    N_MC = 128
    MC_CHUNK = 32
    REFERENCE_BASIS = 60
    SEEDS = [52]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
D = N - 1
A_C = (T - 1j * ETA) / N
TAIL_RADIUS = TAIL_RADIUS_FACTOR * math.sqrt(D)

output_root = Path(os.environ.get("THIMBLE_OUTPUT_ROOT", "runs"))
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = output_root / f"{stamp}_{METHOD}_N{N}"
run_dir.mkdir(parents=True, exist_ok=True)

print(f"method={METHOD}, device={DEVICE}, dtype=float64/complex128")
print(f"N={N}, D={D}, T_c={T}-{ETA}i, lambdas={LAMBDAS}, seeds={SEEDS}")
print(f"epochs={EPOCHS}, batch={BATCH}, N_mc={N_MC}, rank={RANK}")
print(f"tail radius={TAIL_RADIUS:.3f}, output={run_dir.resolve()}")
if QUICK_TEST:
    print("QUICK TEST MODE: these settings are not production results.")


## Physics, exact finite-N reference and safe asymptotic maps

For `lambda > 0`, the quartic term fixes the allowed Stokes sectors. Writing
`a_c = |a_c| exp(-i alpha)`, the lower descending direction connected to the
original real contour is

\[
\theta_{\rm tail}=-\frac{\pi}{8}+\frac{\alpha}{4}.
\]

The competing upper direction is also descending but is not homologous to the
real contour. It is therefore not enough to minimise phase or weight variance:
the correct tail sector must be fixed by construction.


In [ ]:
def complex_action(x, lam):
    """N-slice action with zero endpoints and weight exp(iS)."""
    x = x.to(torch.complex128)
    n = x.shape[0]
    zero = torch.zeros((n, 1), dtype=torch.complex128, device=x.device)
    x_pad = torch.cat((zero, x, zero), dim=1)
    dx = x_pad[:, 1:] - x_pad[:, :-1]
    a = torch.as_tensor(A_C, dtype=torch.complex128, device=x.device)
    kinetic = MASS / (2.0 * a) * torch.sum(dx**2, dim=1)
    harmonic = a * MASS * OMEGA**2 / 2.0 * torch.sum(x**2, dim=1)
    quartic = a * lam / 4.0 * torch.sum(x**4, dim=1)
    return kinetic - harmonic - quartic


def harmonic_maps():
    """Real SPD preconditioner B and exact complex harmonic thimble A0."""
    j = np.arange(1, N)[:, None]
    k = np.arange(1, N)[None, :]
    Q = np.sqrt(2.0 / N) * np.sin(np.pi * j * k / N)
    modes = np.arange(1, N)
    d = (2.0 * MASS / A_C - A_C * MASS * OMEGA**2
         - 2.0 * MASS / A_C * np.cos(np.pi * modes / N))
    B = Q @ np.diag(1.0 / np.sqrt(np.abs(d))) @ Q.T
    A0 = Q @ np.diag(np.sqrt(1j / d)) @ Q.T

    H = np.zeros((D, D), dtype=np.complex128)
    np.fill_diagonal(H, 2.0 * MASS / A_C - A_C * MASS * OMEGA**2)
    np.fill_diagonal(H[1:], -MASS / A_C)
    np.fill_diagonal(H[:, 1:], -MASS / A_C)
    residual = np.max(np.abs(A0.T @ H @ A0 - 1j * np.eye(D)))
    if residual > 1e-10 or np.linalg.det(B) <= 0:
        raise RuntimeError("Harmonic map construction failed")
    return B, A0, residual


B_NP, A0_NP, HARMONIC_MAP_RESIDUAL = harmonic_maps()
ALPHA = -np.angle(A_C)
THETA_LOWER = -np.pi / 8.0 + ALPHA / 4.0

print(f"harmonic-map residual={HARMONIC_MAP_RESIDUAL:.3e}")
print(f"physical lower-tail angle={THETA_LOWER:.6f} rad "
      f"({np.degrees(THETA_LOWER):.3f} degrees)")


def exact_harmonic_lattice():
    H = np.zeros((D, D), dtype=np.complex128)
    np.fill_diagonal(H, 2.0 * MASS / A_C - A_C * MASS * OMEGA**2)
    np.fill_diagonal(H[1:], -MASS / A_C)
    np.fill_diagonal(H[:, 1:], -MASS / A_C)
    pref = (MASS / (2.0 * np.pi * 1j * A_C)) ** (N / 2)
    return pref * (2.0 * np.pi * 1j) ** (D / 2) / np.sqrt(np.linalg.det(H))


def fixed_n_reference(lam, basis_size=REFERENCE_BASIS):
    """Independent evaluation of this exact N-slice discretisation.

    In a truncated harmonic-oscillator basis this evaluates
    <0|F(PF)^(N-1)|0>, where F=exp[-ia p^2/(2m)] and
    P=exp[-ia(x^2/2 + lambda*x^4/4)].
    """
    nb = int(basis_size)
    lowering = np.zeros((nb, nb), dtype=np.float64)
    idx = np.arange(1, nb)
    lowering[idx - 1, idx] = np.sqrt(idx)
    raising = lowering.T
    x_op = (lowering + raising) / np.sqrt(2.0)
    p_op = (lowering - raising) / (1j * np.sqrt(2.0))
    x2 = x_op @ x_op
    x4 = x2 @ x2
    kinetic_operator = p_op @ p_op / (2.0 * MASS)

    ep, Up = scipy.linalg.eigh(kinetic_operator)
    F = (Up * np.exp(-1j * A_C * ep)) @ Up.conj().T
    V = MASS * OMEGA**2 * x2 / 2.0 + lam * x4 / 4.0
    ev, Uv = scipy.linalg.eigh(V)
    P = (Uv * np.exp(-1j * A_C * ev)) @ Uv.conj().T

    phi0 = np.zeros(nb, dtype=np.float64)
    for n in range(0, nb, 2):
        q = n // 2
        log_abs = (0.5 * math.lgamma(2 * q + 1) - q * math.log(2.0)
                   - math.lgamma(q + 1) - 0.25 * math.log(np.pi))
        phi0[n] = (-1.0)**q * math.exp(log_abs)

    evolution = F.copy()
    for _ in range(N - 1):
        evolution = evolution @ P @ F
    return complex(phi0 @ evolution @ phi0)


references = {lam: fixed_n_reference(lam) for lam in LAMBDAS}
harmonic_closed = exact_harmonic_lattice()
if not QUICK_TEST and abs(references[0.0] - harmonic_closed) > 2e-6:
    raise RuntimeError(
        f"Reference basis did not converge at lambda=0: "
        f"basis={references[0.0]}, closed={harmonic_closed}")

with (run_dir / "n16_references.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["N", "lambda", "K_reference_re", "K_reference_im", "abs_K_reference"])
    for lam, value in references.items():
        writer.writerow([N, lam, value.real, value.imag, abs(value)])

print("N=16 fixed-lattice references:")
for lam, value in references.items():
    print(f"  lambda={lam:3.1f}: {value.real:+.10f}{value.imag:+.10f}i  |K|={abs(value):.8f}")


## Tail-safe saddle-anchored flow

The map is

\[
x(z)=A_{\rm tail}z+q(r)\{(A_0-A_{\rm tail})z+C h(z)\},
\qquad q(r)=\exp[-(r/R)^8].
\]

`h` is an odd bottleneck network, so `x(0)=0` exactly. As `r` grows,
`q(r)` vanishes and the contour is forced onto `A_tail`; the optimiser cannot
change this boundary condition. The analytic full Jacobian includes the
derivative of the radial gate.


In [ ]:
class TailSafeFlow(nn.Module):
    def __init__(self, lam, seed):
        super().__init__()
        torch.manual_seed(seed)
        np.random.seed(seed)

        B = torch.tensor(B_NP, dtype=torch.complex128, device=DEVICE)
        A0 = torch.tensor(A0_NP, dtype=torch.complex128, device=DEVICE)
        if lam == 0.0:
            A_tail = A0.clone()  # exact and valid for the quadratic integral
        else:
            A_tail = torch.exp(torch.tensor(1j * THETA_LOWER,
                                             dtype=torch.complex128,
                                             device=DEVICE)) * B

        self.register_buffer("A0", A0)
        self.register_buffer("A_tail", A_tail)
        self.register_buffer("L", A0 - A_tail)
        self.tail_radius = float(TAIL_RADIUS)

        self.W1 = nn.Parameter(torch.randn(RANK, D, device=DEVICE) / math.sqrt(D))
        qmat, _ = torch.linalg.qr(torch.randn(RANK, RANK, device=DEVICE))
        self.W2 = nn.Parameter(qmat)
        # Exactly zero keeps the lambda=0 starting map analytically exact. For
        # lambda>0 the loss immediately supplies gradients to C; W1/W2 begin
        # learning after C has moved away from zero.
        self.C_real = nn.Parameter(torch.zeros(D, RANK, device=DEVICE))
        self.C_imag = nn.Parameter(torch.zeros(D, RANK, device=DEVICE))

    def C(self):
        return self.C_real.to(torch.complex128) + 1j * self.C_imag.to(torch.complex128)

    def features(self, z):
        a1 = torch.tanh(z @ self.W1.T)
        h = torch.tanh(a1 @ self.W2.T)
        return a1, h

    def gate(self, z):
        r2 = torch.sum(z**2, dim=1)
        u = r2 / (self.tail_radius**2)
        half_power = TAIL_POWER // 2
        q = torch.exp(-(u**half_power))
        grad_q = (-(TAIL_POWER / self.tail_radius**2) * q[:, None]
                  * (u[:, None]**(half_power - 1)) * z)
        return q, grad_q

    def forward(self, z, return_cache=False):
        z = z.to(dtype=torch.float64, device=DEVICE)
        a1, h = self.features(z)
        q, grad_q = self.gate(z)
        zc = z.to(torch.complex128)
        C = self.C()
        v = zc @ self.L.T + h.to(torch.complex128) @ C.T
        x = zc @ self.A_tail.T + q[:, None].to(torch.complex128) * v
        if return_cache:
            return x, (a1, h, q, grad_q, v)
        return x

    def jacobian(self, z, cache=None):
        if cache is None:
            _, cache = self.forward(z, return_cache=True)
        a1, h, q, grad_q, v = cache
        da1 = 1.0 - a1**2
        da2 = 1.0 - h**2
        H1 = da1.unsqueeze(-1) * self.W1.unsqueeze(0)              # b,r,D
        H2 = torch.matmul(self.W2.unsqueeze(0), H1)
        H = da2.unsqueeze(-1) * H2                                # dh/dz
        CH = torch.matmul(self.C().unsqueeze(0), H.to(torch.complex128))
        J = (self.A_tail.unsqueeze(0)
             + q[:, None, None].to(torch.complex128) * (self.L.unsqueeze(0) + CH)
             + v.unsqueeze(-1) * grad_q[:, None, :].to(torch.complex128))
        return J


def verify_analytic_jacobian():
    """Finite-difference check of the complete gated Jacobian."""
    model = TailSafeFlow(0.7, 123)
    # Make the nonlinear term appreciable so the check exercises C, both MLP
    # derivatives and the radial-gate derivative—not merely the linear map.
    with torch.no_grad():
        model.C_real.normal_(std=0.03)
        model.C_imag.normal_(std=0.03)
    z = torch.randn(3, D, device=DEVICE)
    x, cache = model(z, return_cache=True)
    J = model.jacobian(z, cache)
    eps = 1e-6
    J_fd = torch.empty_like(J)
    with torch.no_grad():
        for k in range(D):
            dz = torch.zeros_like(z)
            dz[:, k] = eps
            J_fd[:, :, k] = (model(z + dz) - model(z - dz)) / (2.0 * eps)
    err = torch.max(torch.abs(J - J_fd)).item()
    print(f"analytic-Jacobian finite-difference error={err:.3e}")
    if err > 2e-7:
        raise RuntimeError("Analytic Jacobian check failed")


verify_analytic_jacobian()


## Proposal, losses and stable complex importance weights

The phase/descent notebook minimises

\[
\mathbb E[(\operatorname{Re}S)^2]
+\mathbb E[\operatorname{ReLU}(-\operatorname{Im}S)^2].
\]

There is deliberately no sample-dependent division by `abs(S)`, which previously
hid the lambda signal behind the extensive quadratic action.

The log-weight-variance notebook uses the full Jacobian and minimises variance
of `log(abs(W))` plus the circular phase variance. Circular variance avoids the
artificial branch cut introduced by applying ordinary variance to an angle.


In [ ]:
LOG_2PI = math.log(2.0 * math.pi)


def proposal_parameters(lam, training=False):
    if lam == 0.0:
        return (1.0,), (1.0,)
    return (TRAIN_SCALES, TRAIN_PROBS) if training else (MC_SCALES, MC_PROBS)


def sample_mixture(n, dim, scales, probs):
    scales_t = torch.tensor(scales, dtype=torch.float64, device=DEVICE)
    probs_t = torch.tensor(probs, dtype=torch.float64, device=DEVICE)
    component = torch.multinomial(probs_t, n, replacement=True)
    z = torch.randn(n, dim, dtype=torch.float64, device=DEVICE)
    return z * scales_t[component, None]


def mixture_logpdf(z, scales, probs):
    z2 = torch.sum(z**2, dim=1)
    terms = []
    for scale, probability in zip(scales, probs):
        terms.append(math.log(probability) - D * math.log(scale) - 0.5 * D * LOG_2PI
                     - z2 / (2.0 * scale**2))
    return torch.logsumexp(torch.stack(terms, dim=1), dim=1)


def weight_parts(model, z, lam, scales, probs):
    x, cache = model(z, return_cache=True)
    S = complex_action(x, lam)
    J = model.jacobian(z, cache)
    sign_J, logabs_J = torch.linalg.slogdet(J)
    log_P = mixture_logpdf(z, scales, probs)
    log_abs_W = -S.imag + logabs_J - log_P
    phase_W = torch.exp(1j * S.real) * sign_J
    return log_abs_W, phase_W, S, J


def harmonic_estimator_preflight():
    """The exact lambda=0 map must give the lattice answer for every sample."""
    torch.manual_seed(777)
    model = TailSafeFlow(0.0, 777)
    z = torch.randn(256, D, device=DEVICE)
    with torch.no_grad():
        log_abs_W, phase_W, _, _ = weight_parts(model, z, 0.0, (1.0,), (1.0,))
    shift = torch.max(log_abs_W)
    mean_shifted = torch.mean(torch.exp(log_abs_W - shift) * phase_W).cpu().numpy()
    norm = (MASS / (2.0 * np.pi * 1j * A_C)) ** (N / 2)
    K = complex(norm * math.exp(float(shift)) * mean_shifted)
    spread = float(torch.std(log_abs_W).cpu())
    error = abs(K - harmonic_closed)
    print(f"harmonic estimator preflight: K={K:.10f}, error={error:.3e}, "
          f"std(log|W|)={spread:.3e}")
    if error > 2e-10 or spread > 2e-10:
        raise RuntimeError("Harmonic estimator preflight failed")


harmonic_estimator_preflight()


def train_model(lam, seed):
    model = TailSafeFlow(lam, seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    epochs_here = EPOCHS_LAMBDA0 if lam == 0.0 else EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs_here, 1), eta_min=LR * 0.05)
    scales, probs = proposal_parameters(lam, training=True)

    history = {"total": [], "phase": [], "descent": [],
               "logmag_variance": [], "circular_phase": []}
    started = time.time()

    for epoch in range(epochs_here):
        optimizer.zero_grad(set_to_none=True)
        z = sample_mixture(BATCH, D, scales, probs)

        if METHOD == "phase_descent":
            x = model(z)
            S = complex_action(x, lam)
            phase_loss = torch.mean(S.real**2)
            descent_loss = torch.mean(torch.relu(-S.imag)**2)
            loss = phase_loss + descent_loss
            logmag_variance = torch.zeros((), device=DEVICE)
            circular_phase = torch.zeros((), device=DEVICE)
        else:
            log_abs_W, phase_W, S, _ = weight_parts(model, z, lam, scales, probs)
            logmag_variance = torch.var(log_abs_W, unbiased=False)
            circular_phase = 1.0 - torch.abs(torch.mean(phase_W))**2
            loss = logmag_variance + circular_phase
            phase_loss = torch.zeros((), device=DEVICE)
            descent_loss = torch.zeros((), device=DEVICE)

        if not torch.isfinite(loss):
            raise FloatingPointError(f"non-finite training loss at epoch {epoch + 1}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        history["total"].append(float(loss.detach().cpu()))
        history["phase"].append(float(phase_loss.detach().cpu()))
        history["descent"].append(float(descent_loss.detach().cpu()))
        history["logmag_variance"].append(float(logmag_variance.detach().cpu()))
        history["circular_phase"].append(float(circular_phase.detach().cpu()))

        if epoch == 0 or (epoch + 1) % LOG_EVERY == 0 or epoch + 1 == epochs_here:
            if METHOD == "phase_descent":
                detail = (f"phase={history['phase'][-1]:.3e}, "
                          f"descent={history['descent'][-1]:.3e}")
            else:
                detail = (f"logmag-var={history['logmag_variance'][-1]:.3e}, "
                          f"circular-phase={history['circular_phase'][-1]:.3e}")
            print(f"    epoch {epoch+1:5d}/{epochs_here}: loss={history['total'][-1]:.3e}, "
                  f"{detail}, elapsed={time.time()-started:.0f}s", flush=True)

    return model, history, time.time() - started


def estimate(model, lam, seed):
    scales, probs = proposal_parameters(lam, training=False)
    torch.manual_seed(seed + 100_000)
    np.random.seed(seed + 100_000)

    logabs_parts = []
    phase_parts = []
    remaining = N_MC
    model.eval()
    while remaining > 0:
        n_now = min(MC_CHUNK, remaining)
        z = sample_mixture(n_now, D, scales, probs)
        with torch.no_grad():
            log_abs_W, phase_W, _, _ = weight_parts(model, z, lam, scales, probs)
        logabs_parts.append(log_abs_W.cpu())
        phase_parts.append(phase_W.cpu())
        remaining -= n_now

    log_abs_W = torch.cat(logabs_parts)
    phase_W = torch.cat(phase_parts)
    shift = torch.max(log_abs_W)
    shifted = torch.exp(log_abs_W - shift) * phase_W

    norm = (MASS / (2.0 * np.pi * 1j * A_C)) ** (N / 2)
    scale = norm * math.exp(float(shift))
    mean_shifted = torch.mean(shifted).numpy()
    K = complex(scale * mean_shifted)

    centred = shifted - torch.mean(shifted)
    stderr = abs(scale) * math.sqrt(float(torch.mean(torch.abs(centred)**2)) / N_MC)
    ess_over_n = float(torch.abs(torch.mean(shifted))**2
                       / torch.mean(torch.abs(shifted)**2))

    # Four independently useful blocks expose rare-tail instability hidden by one mean.
    block_values = []
    for block in torch.tensor_split(shifted, 4):
        block_values.append(complex(scale * torch.mean(block).numpy()))
    block_scatter = float(np.std(np.asarray(block_values, dtype=np.complex128)))
    return K, stderr, ess_over_n, block_scatter, block_values


def diagnostics(model, lam):
    model.eval()
    with torch.no_grad():
        zero = torch.zeros(1, D, device=DEVICE)
        saddle_error = float(torch.linalg.vector_norm(model(zero)).cpu())

        directions = torch.randn(64, D, device=DEVICE)
        directions = directions / torch.linalg.vector_norm(directions, dim=1, keepdim=True)
        z_tail = directions * (3.0 * TAIL_RADIUS)
        x_tail, cache = model(z_tail, return_cache=True)
        target = z_tail.to(torch.complex128) @ model.A_tail.T
        tail_error = float((torch.linalg.vector_norm(x_tail - target)
                            / torch.linalg.vector_norm(target)).cpu())
        S_tail = complex_action(x_tail, lam)
        min_tail_imS = float(torch.min(S_tail.imag).cpu())

        scales, probs = proposal_parameters(lam, training=False)
        z_j = sample_mixture(256, D, scales, probs)
        _, cache_j = model(z_j, return_cache=True)
        J = model.jacobian(z_j, cache_j)
        min_sigma = float(torch.min(torch.linalg.svdvals(J)[:, -1]).cpu())

    tail_ok = tail_error < 1e-8 and min_tail_imS > 0.0
    return saddle_error, tail_error, min_tail_imS, min_sigma, tail_ok


def plot_losses(history, seed, lam):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    eps = 1e-18
    if METHOD == "phase_descent":
        series = [("total", "Total"), ("phase", "mean[(Re S)^2]"),
                  ("descent", "mean[ReLU(-Im S)^2]")]
    else:
        series = [("total", "Total"), ("logmag_variance", "Var[log |W|]"),
                  ("circular_phase", "Circular phase variance")]
    for ax, (key, title) in zip(axes, series):
        ax.semilogy(np.maximum(np.asarray(history[key]), eps), lw=0.8)
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"{METHOD}: seed={seed}, lambda={lam}, N={N}")
    fig.tight_layout()
    path = run_dir / f"loss_seed{seed}_lambda{lam}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


## Overnight run

Results are appended to CSV after every seed/lambda pair, so completed runs
survive a later failure. A failed run is recorded and the remaining jobs continue.


In [ ]:
csv_path = run_dir / "results.csv"
columns = [
    "status", "method", "seed", "lambda", "N", "epochs", "K_re", "K_im", "abs_K",
    "reference_re", "reference_im", "reference_abs", "relative_complex_error",
    "stderr", "error_over_stderr", "ESS_over_N", "block_scatter", "final_loss",
    "phase_loss", "descent_loss", "logmag_variance", "circular_phase_variance",
    "saddle_error", "tail_relative_error", "min_tail_ImS", "min_sigma_J",
    "tail_check_pass", "time_seconds", "error_message"
]
with csv_path.open("w", newline="", encoding="utf-8") as f:
    csv.DictWriter(f, fieldnames=columns).writeheader()

config = {
    "method": METHOD, "N": N, "T": T, "eta": ETA, "mass": MASS, "omega": OMEGA,
    "lambdas": LAMBDAS, "seeds": SEEDS, "rank": RANK, "epochs": EPOCHS,
    "epochs_lambda0": EPOCHS_LAMBDA0, "batch": BATCH, "learning_rate": LR,
    "N_mc": N_MC, "MC_chunk": MC_CHUNK, "tail_radius": TAIL_RADIUS,
    "tail_power": TAIL_POWER,
    "tail_angle_radians": THETA_LOWER, "reference_basis": REFERENCE_BASIS,
    "device": str(DEVICE), "quick_test": QUICK_TEST,
}
with (run_dir / "config.json").open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

rows = []
for seed in SEEDS:
    for lam in LAMBDAS:
        print("\n" + "=" * 82)
        print(f"{METHOD}: seed={seed}, lambda={lam}, N={N}")
        print("=" * 82, flush=True)
        row = {key: "" for key in columns}
        row.update({"method": METHOD, "seed": seed, "lambda": lam, "N": N})
        try:
            model, history, train_seconds = train_model(lam, seed)
            K, stderr, ess, block_scatter, block_values = estimate(model, lam, seed)
            saddle_error, tail_error, min_tail_imS, min_sigma, tail_ok = diagnostics(model, lam)
            reference = references[lam]
            relative_error = abs(K - reference) / max(abs(reference), 1e-300)
            error_over_stderr = abs(K - reference) / max(stderr, 1e-300)
            block_ok = block_scatter <= max(4.0 * stderr, 0.02 * abs(reference))
            passed = (relative_error <= RELATIVE_ERROR_TOL and tail_ok and block_ok
                      and saddle_error < 1e-12 and np.isfinite(K.real) and np.isfinite(K.imag))
            status = "PASS" if passed else "FAIL_DIAGNOSTICS"

            epochs_here = EPOCHS_LAMBDA0 if lam == 0.0 else EPOCHS
            row.update({
                "status": status, "epochs": epochs_here,
                "K_re": K.real, "K_im": K.imag, "abs_K": abs(K),
                "reference_re": reference.real, "reference_im": reference.imag,
                "reference_abs": abs(reference), "relative_complex_error": relative_error,
                "stderr": stderr, "error_over_stderr": error_over_stderr,
                "ESS_over_N": ess, "block_scatter": block_scatter,
                "final_loss": history["total"][-1], "phase_loss": history["phase"][-1],
                "descent_loss": history["descent"][-1],
                "logmag_variance": history["logmag_variance"][-1],
                "circular_phase_variance": history["circular_phase"][-1],
                "saddle_error": saddle_error, "tail_relative_error": tail_error,
                "min_tail_ImS": min_tail_imS, "min_sigma_J": min_sigma,
                "tail_check_pass": tail_ok, "time_seconds": train_seconds,
            })
            torch.save({"state_dict": model.state_dict(), "config": config, "lambda": lam,
                        "seed": seed}, run_dir / f"model_seed{seed}_lambda{lam}.pt")
            loss_path = plot_losses(history, seed, lam)
            print(f"K_ML  = {K.real:+.10f}{K.imag:+.10f}i  |K|={abs(K):.8f}")
            print(f"K_ref = {reference.real:+.10f}{reference.imag:+.10f}i  "
                  f"relative error={100*relative_error:.3f}%")
            print(f"stderr={stderr:.3e}, ESS/N={ess:.3e}, block scatter={block_scatter:.3e}")
            print(f"tail pass={tail_ok}, min ImS(tail)={min_tail_imS:.3e}, "
                  f"min sigma(J)={min_sigma:.3e}")
            print(f"STATUS: {status}; saved {loss_path}", flush=True)
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
        except Exception as exc:
            row["status"] = "ERROR"
            row["error_message"] = f"{type(exc).__name__}: {exc}"
            print(traceback.format_exc(), flush=True)

        rows.append(row)
        with csv_path.open("a", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=columns).writerow(row)
        with (run_dir / "progress.txt").open("a", encoding="utf-8") as f:
            f.write(f"{datetime.now().isoformat(timespec='seconds')}  "
                    f"seed={seed} lambda={lam} status={row['status']}\n")


In [ ]:
# Combined result and loss-independent summary plot.
successful = [r for r in rows if r["status"] not in ("ERROR", "")]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for seed in SEEDS:
    seed_rows = sorted([r for r in successful if r["seed"] == seed], key=lambda r: r["lambda"])
    if seed_rows:
        axes[0].plot([r["lambda"] for r in seed_rows], [r["K_re"] for r in seed_rows],
                     "o-", label=f"seed {seed} Re")
        axes[1].plot([r["lambda"] for r in seed_rows], [r["K_im"] for r in seed_rows],
                     "s--", label=f"seed {seed} Im")
lams = sorted(references)
axes[0].plot(lams, [references[l].real for l in lams], "k*-", ms=11, label="reference Re")
axes[1].plot(lams, [references[l].imag for l in lams], "k*-", ms=11, label="reference Im")
for ax, component in zip(axes, ("real", "imaginary")):
    ax.axhline(0.0, color="grey", lw=0.6)
    ax.set_xlabel("lambda")
    ax.set_ylabel(f"{component} part of K")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle(f"{METHOD}: N={N} physical-cycle result")
fig.tight_layout()
summary_plot = run_dir / "propagator_vs_lambda.png"
fig.savefig(summary_plot, dpi=150, bbox_inches="tight")
plt.close(fig)

print("\n" + "=" * 82)
print(f"FINISHED {METHOD}")
print(f"CSV: {csv_path.resolve()}")
print(f"Loss curves and summary plot: {run_dir.resolve()}")
print("A neural row is trustworthy only when status is PASS; the independent reference "
      "is retained even when a neural run fails.")
